# 04 — LSTM

Trains an LSTM to predict EV battery capacity from a 128-timestep charging window.

Five configurations are run on fold 0 as an ablation; the selected configuration is
then run across all 5 folds. Every run is logged to MLflow.

Method and results discussion: `docs/LSTM_METHOD.md`

## 1. Setup

In [1]:
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import dagshub
import mlflow

RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

INDEX_DIR = Path("../data/processed")
CACHE_DIR = Path("../data/cache")
FIG_DIR = Path("../reports/figures")
REPORTS_DIR = Path("../reports")
for d in (CACHE_DIR, FIG_DIR, REPORTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Fold assignment fixed in notebook 03. Never regenerated here - the baselines
# were scored on these exact folds, so changing them would void every comparison.
with open(INDEX_DIR / "fold_manifest.json") as fp:
    manifest = json.load(fp)

N_FOLDS = manifest["config"]["n_folds"]
car_to_fold = {int(k): v for k, v in manifest["car_to_fold"].items()}

print(f"Vehicles : {len(car_to_fold)}")
print(f"Folds    : {N_FOLDS}")

Vehicles : 30
Folds    : 5


In [2]:
# Device selection: best available first, CPU last, so the notebook runs
# unmodified on Apple Silicon, NVIDIA, or CPU-only hardware.

if torch.backends.mps.is_available():
    device = torch.device("mps")          # Apple Silicon GPU
elif torch.cuda.is_available():
    device = torch.device("cuda")         # NVIDIA GPU
else:
    device = torch.device("cpu")

# Smoke test: MPS can report available but fail on actual computation.
t = torch.randn(3, 3).to(device)
print(f"device: {device}  |  matmul ok: {(t @ t).shape}")

device: mps  |  matmul ok: torch.Size([3, 3])


In [ ]:
# Reference points, all computed out-of-sample on these same folds (notebook 03).

BASELINE_RMSE_MEAN = 1.877        # mean predictor, no input
BASELINE_RMSE_LINEAR = 1.095      # mileage-only linear regression
BASELINE_STD_LINEAR = 0.21        # fold-to-fold std of the linear baseline
PAPER_LSTM_RMSE = 1.420           # reference paper, their fleet, single split

BASELINE_FOLDS = [1.032, 1.081, 1.379, 0.815, 1.166]   # linear, per fold

BASELINE_BANDS = {                # linear baseline, per mileage band
    "(0, 50]": 0.777,
    "(50, 100]": 0.999,
    "(100, 150]": 0.968,
    "(150, 200]": 1.409,
    "(200, 300]": 1.178,
}

BANDS = [0, 50, 100, 150, 200, 300]

print(f"mean baseline   : {BASELINE_RMSE_MEAN}")
print(f"linear baseline (mileage only): {BASELINE_RMSE_LINEAR} +/- {BASELINE_STD_LINEAR}")
print(f"paper's LSTM    : {PAPER_LSTM_RMSE}")

mean baseline   : 1.877
linear baseline : 1.095 +/- 0.21
paper's LSTM    : 1.42


## 2. Data cache

In [ ]:
def build_cache(n_channels):
    """Load all snippets into memory, cached to .npy.

    Returns X (n, 128, C), y (n,), cars (n,), mil (n,).

    Loading once into RAM rather than reading per batch: 111k snippets is
    ~400-513 MB depending on channel count, which fits comfortably, while
    lazy loading would re-read 111k files every epoch (~26 s of I/O each).

    n_channels = 7  -> raw sensors, timestamp (channel 7) dropped
    n_channels = 9  -> plus cell voltage spread and temperature spread
    """
    suffix = f"_{n_channels}ch"
    files = {k: CACHE_DIR / f"{k}{suffix}.npy" for k in ("X", "y", "cars", "mil")}

    if all(p.exists() for p in files.values()):
        print(f"Loaded {n_channels}-channel cache.")
        return tuple(np.load(files[k]) for k in ("X", "y", "cars", "mil"))

    paths, path_cars = [], []
    for car, car_files in manifest["files"].items():
        paths.extend(car_files)
        path_cars.extend([int(car)] * len(car_files))

    n = len(paths)
    print(f"Building {n_channels}-channel cache from {n:,} snippets...")

    X = np.empty((n, 128, n_channels), dtype=np.float32)
    y = np.empty(n, dtype=np.float32)
    mil = np.empty(n, dtype=np.float32)
    cars = np.asarray(path_cars, dtype=np.int32)

    for i, p in enumerate(tqdm(paths)):
        arr, meta = torch.load(p, weights_only=False)
        raw = arr[:, :7]                        # drop timestamp channel

        if n_channels == 9:
            # Cell imbalance: channels 0/3/4 are near-identical (~4.09-4.13),
            # so the meaningful quantity is their difference. Supplying it
            # directly avoids asking the network to recover a ~0.06 V gap
            # between three separately-standardised inputs.
            v_spread = (raw[:, 3] - raw[:, 4])[:, None]
            t_spread = (raw[:, 5] - raw[:, 6])[:, None]
            X[i] = np.hstack([raw, v_spread, t_spread])
        else:
            X[i] = raw

        y[i] = meta["capacity"]
        mil[i] = meta["mileage"]              # evaluation only, never a model input

    for k, p in files.items():
        np.save(p, {"X": X, "y": y, "cars": cars, "mil": mil}[k])

    return X, y, cars, mil

In [ ]:
# Both caches are built up front. Configs 1-3 use 7 channels, 4-5 use 9.

CACHE = {}
for c in (7, 9):
    X_c, y_c, cars_c, mil_c = build_cache(c)
    folds_c = np.array([car_to_fold[int(v)] for v in cars_c], dtype=np.int8)
    CACHE[c] = {"X": X_c, "y": y_c, "cars": cars_c, "mil": mil_c, "folds": folds_c}
    print(f"  {c}ch: {X_c.shape}  {X_c.nbytes / 1024**2:.0f} MB")

# cars / y / mil / folds are identical across caches; only X differs.
y = CACHE[9]["y"]
cars = CACHE[9]["cars"]
mil = CACHE[9]["mil"]
folds = CACHE[9]["folds"]

print(f"\nsnippets : {len(y):,}")
print(f"capacity : {y.min():.2f} - {y.max():.2f} Ah  "
      f"(mean {y.mean():.2f}, std {y.std():.2f})")
print(f"per fold : {np.bincount(folds)}")

## 3. Splits, scaling, data loading

In [ ]:
def split_indices(test_fold):
    """Return (train_idx, val_idx, test_idx) for a given test fold.

    Three-way split. The network needs a validation set to decide when to stop
    and which checkpoint to keep; using the test fold for those decisions would
    leak test information into training and inflate the reported score.

    Validation rotates with the test fold, so each fold serves as test once,
    validation once, and training three times.

    Cost: sequence models train on 18 vehicles where the baselines used 24.
    """
    val_fold = (test_fold + 1) % N_FOLDS
    test_idx = np.where(folds == test_fold)[0]
    val_idx = np.where(folds == val_fold)[0]
    train_idx = np.where((folds != test_fold) & (folds != val_fold))[0]
    return train_idx, val_idx, test_idx


for f in range(N_FOLDS):
    tr, va, te = split_indices(f)
    print(f"fold {f}:  train {len(tr):>6,} ({len(np.unique(cars[tr])):>2} cars)  "
          f"val {len(va):>6,} ({len(np.unique(cars[va])):>2})  "
          f"test {len(te):>6,} ({len(np.unique(cars[te])):>2})")

In [ ]:
def fit_scaler(X_train):
    """Per-channel mean and std, computed across sample and time axes.

    Fitted on the training split only. Computing statistics over the whole
    dataset would leak test distribution information into training.
    """
    mean = X_train.mean(axis=(0, 1), keepdims=True)
    std = X_train.std(axis=(0, 1), keepdims=True)
    std[std < 1e-8] = 1.0                    # guard constant channels
    return mean, std


class SnippetDataset(Dataset):
    """Charging snippets held in memory, scaled once on construction."""

    def __init__(self, X, y, idx, mean, std, y_mean, y_std, scale_target=True):
        self.X = ((X[idx] - mean) / std).astype(np.float32)

        # scale_target=False reproduces the original configuration, where the
        # target was left in raw Ah. Kept as a flag so config 01 is faithful.
        if scale_target:
            self.y = ((y[idx] - y_mean) / y_std).astype(np.float32)
        else:
            self.y = y[idx].astype(np.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return torch.from_numpy(self.X[i]), torch.tensor(self.y[i])


def make_loaders(test_fold, n_channels, batch_size=256, scale_target=True):
    X = CACHE[n_channels]["X"]
    tr, va, te = split_indices(test_fold)

    mean, std = fit_scaler(X[tr])            # training split only
    y_mean, y_std = float(y[tr].mean()), float(y[tr].std())

    kw = dict(mean=mean, std=std, y_mean=y_mean, y_std=y_std,
              scale_target=scale_target)

    loaders = {
        "train": DataLoader(SnippetDataset(X, y, tr, **kw),
                            batch_size=batch_size, shuffle=True),
        "val": DataLoader(SnippetDataset(X, y, va, **kw),
                          batch_size=batch_size, shuffle=False),
        "test": DataLoader(SnippetDataset(X, y, te, **kw),
                           batch_size=batch_size, shuffle=False),
    }
    return loaders, (y_mean, y_std), (tr, va, te)

## 4. Model

In [ ]:
class LSTMNet(nn.Module):
    """1 LSTM layer -> 2 FC layers. Based on the reference implementation.

    pooling='last'  : use the final hidden state as the sequence summary
    pooling='mean'  : average all 128 hidden states

    dropout_before_output=True reproduces the original head, where dropout sat
    immediately before the regression output. Dropout rescales survivors by
    1/(1-p) to preserve expectation, injecting variance proportional to output
    magnitude - harmful when outputs are ~40. Kept as a flag for config 01.
    """

    def __init__(self, n_features, hidden=128, fc_hidden=64, dropout=0.3,
                 pooling="last", dropout_before_output=False):
        super().__init__()
        self.pooling = pooling

        # batch_first=True -> (batch, seq, feature) instead of (seq, batch, feature)
        self.lstm = nn.LSTM(input_size=n_features, hidden_size=hidden,
                            num_layers=1, batch_first=True)

        if dropout_before_output:
            layers = [nn.Dropout(dropout), nn.Linear(hidden, fc_hidden),
                      nn.LeakyReLU(), nn.Dropout(dropout),
                      nn.Linear(fc_hidden, 1)]
        else:
            layers = [nn.Linear(hidden, fc_hidden), nn.LeakyReLU(),
                      nn.Dropout(dropout), nn.Linear(fc_hidden, 1)]

        self.head = nn.Sequential(*layers)

        # Xavier keeps activation variance stable through the layers
        for m in self.head:
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        out, _ = self.lstm(x)                     # (B, seq, hidden)
        if self.pooling == "mean":
            summary = out.mean(dim=1)
        else:
            summary = out[:, -1, :]               # final timestep
        return self.head(summary).squeeze(-1)     # (B,)

In [ ]:
def metrics(y_true, y_pred):
    """All metrics in Ah.

    corr and std_ratio are diagnostics: a model scoring near the mean baseline
    may be collapsed (constant output, std_ratio ~0) or compressed (tracks
    reality but hedges, std_ratio ~0.3 with corr >0.3). These need different
    fixes, so they are distinguished.
    """
    return {
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "mape": float(np.mean(np.abs((y_true - y_pred) / y_true)) * 100),
        "r2": float(r2_score(y_true, y_pred)),
        "corr": float(np.corrcoef(y_pred, y_true)[0, 1]),
        "std_ratio": float(y_pred.std() / y_true.std()),
    }


def evaluate(model, loader, y_mean, y_std, scale_target=True):
    """Predictions and targets, always returned in Ah."""
    model.eval()                                  # disable dropout
    preds, actuals = [], []

    with torch.no_grad():                         # no gradient tracking needed
        for xb, yb in loader:
            preds.append(model(xb.to(device)).cpu().numpy())
            actuals.append(yb.numpy())

    p, a = np.concatenate(preds), np.concatenate(actuals)

    if scale_target:                              # invert standardisation
        p = p * y_std + y_mean
        a = a * y_std + y_mean

    return p, a

## 5. Training loop

In [ ]:
def train_fold(test_fold, cfg, verbose=False):
    """Train one fold under a configuration dict.

    Returns (model, history, test_metrics, extras).
    """
    loaders, (y_mean, y_std), (tr, va, te) = make_loaders(
        test_fold, cfg["n_features"], cfg["batch_size"], cfg["scale_target"]
    )

    model = LSTMNet(
        n_features=cfg["n_features"],
        hidden=cfg["hidden"],
        fc_hidden=cfg["fc_hidden"],
        dropout=cfg["dropout"],
        pooling=cfg["pooling"],
        dropout_before_output=cfg["dropout_before_output"],
    ).to(device)

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg["lr"],
                                 weight_decay=cfg["weight_decay"])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=4
    )

    history = {"train_loss": [], "val_rmse": [], "lr": []}
    best_val, best_state, stalled = float("inf"), None, 0

    for epoch in range(cfg["epochs"]):
        model.train()                             # enable dropout
        running = 0.0

        for xb, yb in loaders["train"]:
            xb, yb = xb.to(device), yb.to(device)

            optimizer.zero_grad()                 # clear previous gradients
            loss = criterion(model(xb), yb)       # forward + loss
            loss.backward()                       # gradients w.r.t. parameters
            optimizer.step()                      # update parameters

            running += loss.item() * len(yb)

        train_loss = running / len(loaders["train"].dataset)

        vp, va_true = evaluate(model, loaders["val"], y_mean, y_std,
                               cfg["scale_target"])
        val_rmse = float(np.sqrt(mean_squared_error(va_true, vp)))

        scheduler.step(val_rmse)
        history["train_loss"].append(train_loss)
        history["val_rmse"].append(val_rmse)
        history["lr"].append(optimizer.param_groups[0]["lr"])

        # keep the best checkpoint, not the last
        if val_rmse < best_val:
            best_val = val_rmse
            best_state = {k: v.detach().clone()
                          for k, v in model.state_dict().items()}
            stalled = 0
        else:
            stalled += 1

        if verbose:
            print(f"  epoch {epoch+1:>3}  train {train_loss:>9.4f}  "
                  f"val_rmse {val_rmse:.4f}"
                  f"{'  *' if stalled == 0 else ''}")

        if stalled >= cfg["patience"]:
            break

    model.load_state_dict(best_state)

    tp, ta = evaluate(model, loaders["test"], y_mean, y_std, cfg["scale_target"])

    extras = {"test_idx": te, "test_pred": tp, "test_actual": ta,
              "epochs_run": len(history["train_loss"]), "best_val_rmse": best_val}
    return model, history, metrics(ta, tp), extras

In [ ]:
def band_metrics(extras):
    """Per-mileage-band metrics on the test fold.

    Aggregate RMSE is dominated by the 100-150k band (41% of snippets), so a
    band breakdown is needed to see where a model actually helps.
    """
    te_mil = mil[extras["test_idx"]] / 1000
    cut = pd.cut(te_mil, bins=BANDS)
    rows = []

    for band in cut.categories:
        m = cut == band
        if m.sum() < 50:                  # too few to be meaningful
            continue
        rows.append({"band": str(band), "n": int(m.sum()),
                     **metrics(extras["test_actual"][m], extras["test_pred"][m])})

    return rows

## 6. MLflow

In [ ]:
dagshub.init(repo_owner="RutikaKadam10",
             repo_name="ev-battery-capacity-prediction",
             mlflow=True)

mlflow.set_experiment("lstm")
print("tracking to:", mlflow.get_tracking_uri())

In [ ]:
def log_run(cfg, met, extras, history, bands=None, tags=None, checkpoint=None):
    """Log one training run to the active MLflow run."""
    mlflow.log_params(cfg)

    # per-epoch curves; step= makes them render as curves in the UI
    for ep, (tl, vr, lr_) in enumerate(zip(history["train_loss"],
                                           history["val_rmse"],
                                           history["lr"])):
        mlflow.log_metric("train_loss", tl, step=ep)
        mlflow.log_metric("val_rmse", vr, step=ep)
        mlflow.log_metric("learning_rate", lr_, step=ep)

    mlflow.log_metrics(met)
    mlflow.log_metric("epochs_run", extras["epochs_run"])
    mlflow.log_metric("best_val_rmse", extras["best_val_rmse"])

    # gap over the mean baseline, and that gap relative to evaluation noise
    mlflow.log_metric("gap_vs_mean_baseline", BASELINE_RMSE_MEAN - met["rmse"])
    mlflow.log_metric("gap_vs_linear_baseline", BASELINE_RMSE_LINEAR - met["rmse"])

    if bands:
        for b in bands:
            key = b["band"].replace("(", "").replace("]", "")
            key = key.replace(", ", "_").replace(" ", "")
            mlflow.log_metric(f"rmse_band_{key}", b["rmse"])
            mlflow.log_metric(f"n_band_{key}", b["n"])

    if tags:
        mlflow.set_tags(tags)

    if checkpoint is not None:
        mlflow.log_artifact(str(checkpoint))

## 7. Ablation on fold 0

In [ ]:
BASE = {
    "model": "LSTM",
    "fc_hidden": 64,
    "batch_size": 256,
    "weight_decay": 1e-5,
    "optimizer": "Adam",
    "scheduler": "ReduceLROnPlateau",
    "seq_len": 128,
    "n_cars": len(car_to_fold),
    "n_folds": N_FOLDS,
    "seed": RANDOM_SEED,
    "timestamp_channel_dropped": True,
    "mileage_as_input": False,
}


def cfg(**kw):
    """Build a full config from BASE plus overrides."""
    c = dict(BASE)
    c.update({
        "n_features": 9, "hidden": 128, "dropout": 0.3, "lr": 3e-4,
        "epochs": 80, "patience": 10, "pooling": "last",
        "scale_target": True, "dropout_before_output": False,
    })
    c.update(kw)
    return c


# Five configurations, in the order they were developed.
ABLATIONS = [
    ("01-unscaled-target",
     cfg(n_features=7, lr=1e-3, epochs=40, patience=6,
         scale_target=False, dropout_before_output=True),
     {"change": "original configuration",
      "hypothesis": "baseline attempt"}),

    ("02-scaled-target",
     cfg(n_features=7, lr=1e-3, epochs=40, patience=6),
     {"change": "standardise target; move dropout off the output layer",
      "hypothesis": "raw ~40 Ah target forces a large constant offset and "
                    "amplifies dropout noise at the output"}),

    ("03-mean-pooling",
     cfg(n_features=7, pooling="mean"),
     {"change": "mean-pool all 128 hidden states instead of the last",
      "hypothesis": "the final hidden state discards 127 of 128 states"}),

    ("04-spread-features",
     cfg(n_features=9),
     {"change": "add cell voltage and temperature spread channels",
      "hypothesis": "channels 0/3/4 are near-identical; their difference "
                    "(cell imbalance) is the degradation-relevant quantity"}),

    ("05-regularised",
     cfg(n_features=9, hidden=32, dropout=0.5),
     {"change": "hidden 128 -> 32, dropout 0.3 -> 0.5",
      "hypothesis": "validation peaked at epoch 1 while train loss kept "
                    "falling: overfitting 18 vehicles immediately"}),
]

for name, c, notes in ABLATIONS:
    print(f"{name:<22} {c['n_features']}ch  hidden {c['hidden']:<4} "
          f"drop {c['dropout']}  lr {c['lr']:<7} pool {c['pooling']:<5} "
          f"scaled_y {c['scale_target']}")

In [ ]:
# Each configuration trained on fold 0 and logged as a nested MLflow run.

ablation_results = []
ablation_histories = {}

with mlflow.start_run(run_name="lstm-ablation-fold0") as parent:
    mlflow.set_tags({
        "scope": "fold 0 only",
        "purpose": "configuration selection",
    })

    for name, c, notes in ABLATIONS:
        print(f"\n{'='*64}\n{name}\n{'='*64}")

        with mlflow.start_run(run_name=name, nested=True):
            t0 = time.time()
            model, hist, met, extras = train_fold(0, c, verbose=False)
            elapsed = time.time() - t0

            log_run(cfg={**c, "fold": 0}, met=met, extras=extras,
                    history=hist, bands=band_metrics(extras),
                    tags={**notes, "ablation": name})
            mlflow.log_metric("train_seconds", elapsed)

            ablation_results.append({"config": name, **met,
                                     "epochs": extras["epochs_run"],
                                     "seconds": elapsed})
            ablation_histories[name] = hist

            print(f"  RMSE {met['rmse']:.4f}  R2 {met['r2']:>7.4f}  "
                  f"corr {met['corr']:.3f}  std_ratio {met['std_ratio']:.3f}  "
                  f"({elapsed:.0f}s, {extras['epochs_run']} epochs)")

    # summary table on the parent run
    abl = pd.DataFrame(ablation_results)
    mlflow.log_metric("best_rmse", abl["rmse"].min())
    abl.to_csv(REPORTS_DIR / "lstm_ablation.csv", index=False)
    mlflow.log_artifact(str(REPORTS_DIR / "lstm_ablation.csv"))

print("\n" + "=" * 64)
print(abl.round(4).to_string(index=False))
print("=" * 64)
print(f"fold 0 linear baseline: RMSE {BASELINE_FOLDS[0]:.3f}")

In [ ]:
# Mark the selected configuration so the choice is visible in the run history.

best_name = abl.loc[abl["rmse"].idxmin(), "config"]
print(f"lowest RMSE on fold 0: {best_name}")

runs = mlflow.search_runs(
    experiment_names=["lstm"],
    filter_string="tags.ablation != ''",
)

client = mlflow.tracking.MlflowClient()
for _, r in runs.iterrows():
    tag = r.get("tags.ablation")
    if not isinstance(tag, str):
        continue
    status = "selected" if tag == best_name else (
        "rejected" if tag == "03-mean-pooling" else "superseded")
    client.set_tag(r["run_id"], "status", status)

print("status tags written")

In [ ]:
# Ablation comparison plot

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
names = abl["config"].tolist()
xpos = range(len(names))

axes[0].bar(xpos, abl["rmse"], color="steelblue")
axes[0].axhline(BASELINE_FOLDS[0], color="darkorange", ls="--",
                label=f"linear baseline ({BASELINE_FOLDS[0]})")
axes[0].axhline(BASELINE_RMSE_MEAN, color="grey", ls=":", label="mean baseline")
axes[0].set_xticks(xpos); axes[0].set_xticklabels(names, rotation=30, ha="right")
axes[0].set_ylabel("RMSE (Ah)"); axes[0].set_title("Test RMSE, fold 0")
axes[0].legend(fontsize=8)

axes[1].bar(xpos, abl["corr"], color="seagreen")
axes[1].set_xticks(xpos); axes[1].set_xticklabels(names, rotation=30, ha="right")
axes[1].set_ylabel("corr(pred, actual)"); axes[1].set_title("Signal captured")

axes[2].bar(xpos, abl["std_ratio"], color="indianred")
axes[2].axhline(1.0, color="black", ls=":", lw=1, label="matches reality")
axes[2].set_xticks(xpos); axes[2].set_xticklabels(names, rotation=30, ha="right")
axes[2].set_ylabel("pred std / actual std"); axes[2].set_title("Commitment")
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig(FIG_DIR / "lstm_ablation.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Loss curves for every ablation, on shared axes

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

for name, h in ablation_histories.items():
    # config 01 trains on an unscaled target, so its loss is in Ah^2 and
    # not comparable; plotted separately below if needed
    if name != "01-unscaled-target":
        axes[0].plot(h["train_loss"], label=name)
    axes[1].plot(h["val_rmse"], label=name)

axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Train MSE (scaled units)")
axes[0].set_title("Training loss"); axes[0].legend(fontsize=8)

axes[1].axhline(BASELINE_FOLDS[0], color="darkorange", ls="--", lw=1,
                label="linear baseline")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Validation RMSE (Ah)")
axes[1].set_title("Validation RMSE"); axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(FIG_DIR / "lstm_ablation_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Selected configuration, full 5-fold

In [ ]:
FINAL = cfg(n_features=9, hidden=32, dropout=0.5)

fold_results, band_rows, fold_histories = [], [], {}

with mlflow.start_run(run_name="lstm-final-5fold") as parent:
    mlflow.log_params(FINAL)
    mlflow.set_tags({"scope": "all folds", "status": "final"})

    for fold in range(N_FOLDS):
        print(f"\n{'='*64}\nFOLD {fold}\n{'='*64}")

        with mlflow.start_run(run_name=f"fold-{fold}", nested=True):
            t0 = time.time()
            model, hist, met, extras = train_fold(fold, FINAL, verbose=False)
            elapsed = time.time() - t0

            ckpt = REPORTS_DIR / f"lstm_fold{fold}.pt"
            torch.save(model.state_dict(), ckpt)

            bands = band_metrics(extras)
            log_run(cfg={**FINAL, "fold": fold}, met=met, extras=extras,
                    history=hist, bands=bands, checkpoint=ckpt)
            mlflow.log_metric("train_seconds", elapsed)

            for b in bands:
                band_rows.append({"fold": fold, **b})

            fold_results.append({"fold": fold, **met,
                                 "epochs": extras["epochs_run"],
                                 "seconds": elapsed})
            fold_histories[fold] = hist

            print(f"  RMSE {met['rmse']:.4f}  R2 {met['r2']:>7.4f}  "
                  f"corr {met['corr']:.3f}  ({elapsed:.0f}s, "
                  f"{extras['epochs_run']} epochs)")

    res = pd.DataFrame(fold_results)

    # cross-fold aggregates on the parent run
    for m in ("rmse", "mae", "mape", "r2", "corr", "std_ratio"):
        mlflow.log_metric(f"{m}_mean", res[m].mean())
        mlflow.log_metric(f"{m}_std", res[m].std())

    # gap over the mean baseline relative to fold-to-fold noise:
    # a gap smaller than the std cannot be claimed as a real improvement
    gap = BASELINE_RMSE_MEAN - res["rmse"].mean()
    mlflow.log_metric("gap_vs_mean_baseline", gap)
    mlflow.log_metric("gap_to_noise_ratio", gap / res["rmse"].std())

    res.to_csv(REPORTS_DIR / "lstm_per_fold.csv", index=False)
    bands_df = pd.DataFrame(band_rows)
    bands_df.to_csv(REPORTS_DIR / "lstm_per_band.csv", index=False)
    mlflow.log_artifact(str(REPORTS_DIR / "lstm_per_fold.csv"))
    mlflow.log_artifact(str(REPORTS_DIR / "lstm_per_band.csv"))

print("\n" + "=" * 64)
print(res.round(4).to_string(index=False))
print("=" * 64)
print(f"LSTM            : RMSE {res['rmse'].mean():.3f} ± {res['rmse'].std():.3f}")
print(f"linear baseline : RMSE {BASELINE_RMSE_LINEAR} ± {BASELINE_STD_LINEAR}")
print(f"mean baseline   : RMSE {BASELINE_RMSE_MEAN}")
print(f"paper's LSTM    : RMSE {PAPER_LSTM_RMSE}")
print(f"\ngap over mean baseline : {gap:.3f}")
print(f"gap / fold std         : {gap / res['rmse'].std():.1f}x  "
      f"(>2 to claim as real)")

In [ ]:
# Per-band comparison against the linear baseline

band_agg = bands_df.groupby("band").agg(
    folds=("fold", "nunique"),
    n_total=("n", "sum"),
    lstm_rmse=("rmse", "mean"),
    lstm_std=("rmse", "std"),
).round(3)

band_agg["baseline_rmse"] = [BASELINE_BANDS.get(b, np.nan) for b in band_agg.index]
band_agg["lstm_minus_baseline"] = (band_agg["lstm_rmse"]
                                   - band_agg["baseline_rmse"]).round(3)

band_agg = band_agg.reindex(["(0, 50]", "(50, 100]", "(100, 150]",
                            "(150, 200]", "(200, 300]"])
band_agg

In [ ]:
# Final comparison plot

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))

axes[0].plot(res["fold"], res["rmse"], marker="D", color="seagreen", label="LSTM")
axes[0].plot(range(N_FOLDS), BASELINE_FOLDS, marker="s", color="darkorange",
             label="Linear (mileage)")
axes[0].axhline(PAPER_LSTM_RMSE, color="crimson", ls="--", lw=1,
                label="Reference paper LSTM")
axes[0].axhline(BASELINE_RMSE_MEAN, color="grey", ls=":", lw=1,
                label="Mean baseline")
axes[0].axhspan(BASELINE_RMSE_LINEAR - BASELINE_STD_LINEAR,
                BASELINE_RMSE_LINEAR + BASELINE_STD_LINEAR,
                color="darkorange", alpha=0.08, label="baseline ±1 std")
axes[0].set_xlabel("Fold"); axes[0].set_ylabel("RMSE (Ah)")
axes[0].set_xticks(range(N_FOLDS))
axes[0].set_title("By fold"); axes[0].legend(fontsize=8)

w = 0.38
idx = np.arange(len(band_agg))
axes[1].bar(idx - w/2, band_agg["baseline_rmse"], w, label="Linear (mileage)",
            color="darkorange")
axes[1].bar(idx + w/2, band_agg["lstm_rmse"], w, label="LSTM", color="seagreen")
axes[1].set_xticks(idx)
axes[1].set_xticklabels(band_agg.index, rotation=20, ha="right")
axes[1].set_xlabel("Mileage band (thousand km)"); axes[1].set_ylabel("RMSE (Ah)")
axes[1].set_title("By mileage band"); axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(FIG_DIR / "lstm_vs_baseline.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Diagnostics: is the model collapsed, or tracking-but-hedging?

print(res[["fold", "rmse", "r2", "corr", "std_ratio"]].round(3)
      .to_string(index=False))

print(f"\nmean corr      : {res['corr'].mean():.3f}")
print(f"mean std_ratio : {res['std_ratio'].mean():.3f}")
print()
print("  std_ratio < 0.2              -> collapsed, optimisation problem")
print("  std_ratio 0.3-0.5, corr <0.2 -> varying but no signal found")
print("  std_ratio 0.3-0.5, corr >0.3 -> real signal, under-committing")
print("  std_ratio ~1.0               -> healthy")

In [ ]:
# Final summary

summary = pd.DataFrame([
    {"model": "Mean baseline", "inputs": "none",
     "rmse": BASELINE_RMSE_MEAN, "r2": -0.052},
    {"model": "Linear regression", "inputs": "mileage",
     "rmse": BASELINE_RMSE_LINEAR, "r2": 0.635},
    {"model": "LSTM", "inputs": f"128x{FINAL['n_features']} sequence",
     "rmse": round(res["rmse"].mean(), 3), "r2": round(res["r2"].mean(), 3)},
    {"model": "Reference paper LSTM", "inputs": "128x8 sequence",
     "rmse": PAPER_LSTM_RMSE, "r2": np.nan},
])

summary.to_csv(REPORTS_DIR / "lstm_summary.csv", index=False)
summary